In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Load data
stats = pd.read_csv('../data/processed/advanced_stats_clean.csv')
print(stats.shape)
stats.head()

(5441, 59)


,Player,Age,Team,Pos,G,GS,MP,PER,TS%,3PAr,...,19.6,3.7,2.0,5.7,.090,0.4,-0.9,-0.5,1.2.1,bridgmi01
0,Kevin Durant,21.0,OKC,SF,82.0,82.0,3239.0,26.2,0.607,0.210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andre Iguodala,26.0,PHI,SF,82.0,82.0,3193.0,17.8,0.535,0.271,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Rudy Gay,23.0,MEM,SF,80.0,80.0,3175.0,16.2,0.535,0.157,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Stephen Jackson,31.0,2TM,SG,81.0,81.0,3129.0,15.6,0.518,0.277,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Stephen Jackson,31.0,GSW,PF,9.0,9.0,300.0,14.5,0.499,0.301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Availability score (injury proxy)
stats['availability_score'] = (stats['G'] / 82) * 100
stats['availability_score'] = stats['availability_score'].clip(0, 100)
stats[['Player', 'Season', 'G', 'availability_score']].head()

,Player,Season,G,availability_score
0,Kevin Durant,2010,82.0,100.000000
1,Andre Iguodala,2010,82.0,100.000000
2,Rudy Gay,2010,80.0,97.560976
3,Stephen Jackson,2010,81.0,98.780488
4,Stephen Jackson,2010,9.0,10.975610


In [ ]:
#Age score (peaks at 27)
def age_score(age):
    peak = 27
    if age <= peak:
        return max(0, 100 - (peak - age) * 5)
    else:
        return max(0, 100 - (age - peak) * 7)

stats['age_score'] = stats['Age'].apply(age_score)
stats[['Player', 'Season', 'Age', 'age_score']].head()

,Player,Season,Age,age_score
0,Kevin Durant,2010,21.0,70.0
1,Andre Iguodala,2010,26.0,95.0
2,Rudy Gay,2010,23.0,80.0
3,Stephen Jackson,2010,31.0,72.0
4,Stephen Jackson,2010,31.0,72.0


In [ ]:
#Consistency score
stats['consistency_score'] = (stats['GS'] / stats['G'].replace(0, np.nan)) * 100
stats['consistency_score'] = stats['consistency_score'].fillna(0).clip(0, 100)
stats[['Player', 'Season', 'GS', 'G', 'consistency_score']].head()

,Player,Season,GS,G,consistency_score
0,Kevin Durant,2010,82.0,82.0,100.0
1,Andre Iguodala,2010,82.0,82.0,100.0
2,Rudy Gay,2010,80.0,80.0,100.0
3,Stephen Jackson,2010,81.0,81.0,100.0
4,Stephen Jackson,2010,9.0,9.0,100.0


In [9]:
#Offensive and defensive contribution
stats['off_contribution'] = stats['OBPM'] + stats['OWS']
stats['def_contribution'] = stats['DBPM'] + stats['DWS']
stats[['Player', 'Season', 'off_contribution', 'def_contribution']].head()

,Player,Season,off_contribution,def_contribution
0,Kevin Durant,2010,17.4,5.8
1,Andre Iguodala,2010,6.0,3.2
2,Rudy Gay,2010,4.8,2.1
3,Stephen Jackson,2010,0.8,4.7
4,Stephen Jackson,2010,0.1,-1.3


In [10]:
#Composite raw score
stats['raw_score'] = (
    stats['VORP']               * 0.20 +
    stats['BPM']                * 0.15 +
    stats['WS']                 * 0.15 +
    stats['PER']                * 0.10 +
    stats['availability_score'] * 0.15 +
    stats['age_score']          * 0.15 +
    stats['consistency_score']  * 0.10
)
stats[['Player', 'Season', 'raw_score']].head()

,Player,Season,raw_score
0,Kevin Durant,2010,43.100000
1,Andre Iguodala,2010,43.095000
2,Rudy Gay,2010,39.689146
3,Stephen Jackson,2010,38.367073
4,Stephen Jackson,2010,23.721341


In [ ]:
# Normalize to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))
stats['value_score'] = scaler.fit_transform(stats[['raw_score']])
stats['value_score'] = stats['value_score'].round(1)
stats[['Player', 'Season', 'raw_score', 'value_score']].head()

,Player,Season,raw_score,value_score
0,Kevin Durant,2010,43.100000,70.9
1,Andre Iguodala,2010,43.095000,70.9
2,Rudy Gay,2010,39.689146,65.8
3,Stephen Jackson,2010,38.367073,63.9
4,Stephen Jackson,2010,23.721341,42.1


In [ ]:
#Sanity check — top 10 players 2024
top = stats[stats['Season'] == 2024].nlargest(10, 'value_score')[['Player', 'Age', 'Team', 'value_score', 'availability_score', 'age_score']]
print(top.to_string())

                  Player   Age Team  value_score  availability_score  age_score
5065       Pascal Siakam  29.0  2TM         68.3           97.560976       86.0
5074    Donte DiVincenzo  27.0  NYK         68.3           98.780488      100.0
5068       Grayson Allen  28.0  PHO         67.1           91.463415       93.0
5076       Malik Beasley  27.0  MIL         67.1           96.341463      100.0
5064       Fred VanVleet  29.0  HOU         66.9           89.024390       86.0
5072       Chet Holmgren  21.0  OKC         66.4          100.000000       70.0
5090        Jusuf Nurkić  29.0  PHO         66.2           92.682927       86.0
5100          Tyus Jones  27.0  WAS         66.0           80.487805      100.0
5078           Max Strus  27.0  CLE         65.8           85.365854      100.0
5121  Kristaps Porziņģis  28.0  BOS         65.2           69.512195       93.0
